In [38]:
# Cell 02: Imports
import sqlite3
import hashlib
import random
import time
import threading
from datetime import datetime

import ipywidgets as w
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

In [39]:
# Cell 03: Global State
APP_STATE = {
    "logged_in": False,
    "user": None,
    "role": None,
    "radar_running": False,
    "radar_thread": None,
    "threats": [],
    "logs": []
}

ALERT_LEVELS = ["GREEN", "YELLOW", "ORANGE", "RED"]

In [40]:
# Cell 04: SQLite In-Memory DB
conn = sqlite3.connect(":memory:", check_same_thread=False)
cur = conn.cursor()

In [41]:
# Cell 05: Create Users Table
cur.execute("""
CREATE TABLE users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE,
    password_hash TEXT,
    role TEXT
)
""")
conn.commit()

In [42]:
# Cell 06: Hash Function
def hash_password(pw: str) -> str:
    return hashlib.sha256(pw.encode("utf-8")).hexdigest()

In [43]:
# Cell 07: Insert Default Users
DEFAULT_USERS = [
    ("admin", hash_password("admin123"), "Admin"),
    ("rajendra789", hash_password("fuck"), "Operator")
]

for user, pw_hash, role in DEFAULT_USERS:
    cur.execute("INSERT INTO users (username, password_hash, role) VALUES (?, ?, ?)", (user, pw_hash, role))
conn.commit()

In [44]:
# Cell 08: Login Widgets
login_user = w.Text(description="Username", layout=w.Layout(width="250px"))
login_pass = w.Password(description="Password", layout=w.Layout(width="250px"))
login_btn = w.Button(description="Login", button_style="primary")

login_msg = w.Output()

In [45]:
# Cell 09: Login UI Display
login_box = w.VBox([login_user, login_pass, login_btn, login_msg])

In [46]:
# Cell 10: Authentication Function
def authenticate(username, password):
    pw_hash = hash_password(password)
    cur.execute("SELECT username, role FROM users WHERE username=? AND password_hash=?", (username, pw_hash))
    row = cur.fetchone()
    if row:
        return {"username": row[0], "role": row[1]}
    return None

In [47]:
# Cell 11: Login Handler
def show_dashboard():
    pass


def on_login_click(_):
    with login_msg:
        clear_output()
        user = authenticate(login_user.value.strip(), login_pass.value.strip())
        if user:
            APP_STATE["logged_in"] = True
            APP_STATE["user"] = user["username"]
            APP_STATE["role"] = user["role"]
            print(f"✅ Login successful. Role: {user['role']}")
            show_dashboard()
        else:
            print("❌ Invalid credentials.")

login_btn.on_click(on_login_click)

In [48]:
# Cell 12: Session Reset
def stop_radar():
    pass


def logout():
    APP_STATE["logged_in"] = False
    APP_STATE["user"] = None
    APP_STATE["role"] = None
    stop_radar()
    display(login_box)

In [49]:
# Cell 13: Dashboard Containers
dash_header = w.HTML()
dash_alert = w.HTML()
dash_radar_out = w.Output()
dash_log_out = w.Output()
dash_cmd_out = w.Output()

dash_tabs = w.Tab()

In [50]:
# Cell 14: Threat Simulation Engine (Base)
def generate_threat():
    threat_types = ["Air", "Naval", "Cyber", "Satellite", "Land", "Missile", "Drone"]
    return {
        "type": random.choice(threat_types),
        "level": random.randint(1, 10),
        "timestamp": datetime.utcnow().isoformat()
    }

In [51]:
# Cell 15: Threat Generator
def log_event(param):
    pass


def add_threat():
    threat = generate_threat()
    APP_STATE["threats"].append(threat)
    log_event(f"Threat detected: {threat['type']} (L{threat['level']})")

In [52]:
# Cell 16: Alert Level Calculation
def get_alert_level():
    if not APP_STATE["threats"]:
        return "GREEN"
    avg = sum(t["level"] for t in APP_STATE["threats"]) / len(APP_STATE["threats"])
    if avg < 3: return "GREEN"
    if avg < 5: return "YELLOW"
    if avg < 7: return "ORANGE"
    return "RED"

In [53]:
# Cell 17: Logging
def log_event(msg):
    APP_STATE["logs"].append(f"{datetime.utcnow().isoformat()} | {msg}")

In [54]:
# Cell 18: Activity History Display
def refresh_logs():
    with dash_log_out:
        clear_output()
        for line in APP_STATE["logs"][-15:]:
            print(line)

In [55]:
# Cell 19: Command Panel Widgets
cmd_btn_scan = w.Button(description="Run Scan", button_style="info")
cmd_btn_lock = w.Button(description="Lock Down", button_style="danger")
cmd_btn_clear = w.Button(description="Clear Threats", button_style="warning")
cmd_btn_logout = w.Button(description="Logout", button_style="")

cmd_panel = w.VBox([cmd_btn_scan, cmd_btn_lock, cmd_btn_clear, cmd_btn_logout, dash_cmd_out])

In [56]:
# Cell 20: Command Handlers
def refresh_alert():
    pass


def cmd_scan(_):
    add_threat()
    refresh_alert()
    refresh_logs()

def cmd_lock(_):
    log_event("Emergency lock-down issued by command.")
    refresh_logs()

def cmd_clear(_):
    APP_STATE["threats"].clear()
    log_event("Threat list cleared.")
    refresh_alert()
    refresh_logs()

def cmd_logout(_):
    logout()

cmd_btn_scan.on_click(cmd_scan)
cmd_btn_lock.on_click(cmd_lock)
cmd_btn_clear.on_click(cmd_clear)
cmd_btn_logout.on_click(cmd_logout)

In [57]:
# Cell 21: Radar Setup
plt.ioff()

In [58]:
# Cell 22: Radar Plot Update
def render_radar():
    with dash_radar_out:
        clear_output()
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.set_title("Radar Simulation")
        ax.set_xlim(-10, 10)
        ax.set_ylim(-10, 10)
        ax.set_facecolor("black")
        ax.tick_params(colors="white")

        for t in APP_STATE["threats"][-12:]:
            x, y = random.uniform(-10, 10), random.uniform(-10, 10)
            ax.scatter(x, y, c="lime", s=40)

        plt.show()

In [59]:
# Cell 23: Radar Start/Stop
def radar_loop():
    while APP_STATE["radar_running"]:
        render_radar()
        time.sleep(2)

def start_radar():
    if not APP_STATE["radar_running"]:
        APP_STATE["radar_running"] = True
        APP_STATE["radar_thread"] = threading.Thread(target=radar_loop, daemon=True)
        APP_STATE["radar_thread"].start()

def stop_radar():
    APP_STATE["radar_running"] = False

In [60]:
# Cell 24: Alert Indicator
def refresh_alert():
    level = get_alert_level()
    color = {"GREEN":"#00ff00","YELLOW":"#ffff00","ORANGE":"#ff9900","RED":"#ff0000"}[level]
    dash_alert.value = f"<h3 style='color:{color}'>ALERT LEVEL: {level}</h3>"

In [61]:
# Cell 25: Dashboard Render
def show_dashboard():
    dash_header.value = f"<h2>Welcome {APP_STATE['user']} ({APP_STATE['role']})</h2>"
    refresh_alert()
    refresh_logs()
    start_radar()
    dash_tabs.children = [dash_radar_out, cmd_panel, dash_log_out]
    dash_tabs.set_title(0, "Radar")
    dash_tabs.set_title(1, "Commands")
    dash_tabs.set_title(2, "Activity Log")
    display(w.VBox([dash_header, dash_alert, dash_tabs]))

In [62]:
# Cell 26: Role-Based Access
def apply_role_rules():
    role = APP_STATE["role"]
    if role == "Operator":
        cmd_btn_lock.disabled = True
    else:
        cmd_btn_lock.disabled = False

In [63]:
# Cell 27: Threat Module - Air
def module_air():
    add_threat()
    log_event("Air module ping executed.")

In [64]:
# Cell 28: Threat Module - Naval
def module_naval():
    add_threat()
    log_event("Naval module sonar sweep.")

In [65]:
# Cell 29: Threat Module - Cyber
def module_cyber():
    add_threat()
    log_event("Cyber intrusion check completed.")

In [66]:
# Cell 30: Threat Module - Satellite
def module_satellite():
    add_threat()
    log_event("Satellite tracking sync performed.")

In [67]:
# Cell 31: Threat Module - Land
def module_land():
    add_threat()
    log_event("Land perimeter sensors triggered.")

In [68]:
# Cell 32: Threat Module - Missile
def module_missile():
    add_threat()
    log_event("Missile shield diagnostics done.")

In [69]:
# Cell 33: Threat Module - Drone
def module_drone():
    add_threat()
    log_event("Drone sweep initiated.")

In [70]:
# Cell 34: Threat Module - Intel
def module_intel():
    add_threat()
    log_event("Intel data pipeline refreshed.")

In [71]:
# Cell 35: Threat Module - Comms
def module_comms():
    add_threat()
    log_event("Secure comms test completed.")

In [72]:
# Cell 36: Threat Module - Energy
def module_energy():
    add_threat()
    log_event("Power grid anomaly detected.")

In [73]:
# Cell 37: Threat Module - Emergency Drill
def module_drill():
    add_threat()
    log_event("Emergency drill executed.")

In [75]:
# Cell 38: Launch
display(login_box)